## Start pyspark

In [0]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("movielens-local-bronze")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")


## Define schemas

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, LongType,StringType

movies_schema = StructType([
    StructField("movieId", IntegerType(), False),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True),
])

links_schema = StructType([
    StructField("movieId", IntegerType(), False),
    StructField("imdbId", IntegerType(), True),
    StructField("tmdbId", IntegerType(), True),
])

tags_schema = StructType([
    StructField("userId", IntegerType(), False),
    StructField("movieId", IntegerType(), False),
    StructField("tag", StringType(), True),
    StructField("timestamp", LongType(), True),
])
ratings_schema = StructType([
    StructField("userId", IntegerType(), False),
    StructField("movieId", IntegerType(), False),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", LongType(), True),  # unix epoch seconds
])


## Read data sets

In [0]:
movies_path   = r"..\data\raw\movies.csv"
links_path    = r"..\data\raw\links.csv"
tags_path     = r"..\data\raw\tags.csv"
ratings_path  = r"..\data\raw\ratings.csv"

df_movies_raw  = spark.read.schema(movies_schema).option("header", True).csv(movies_path)
df_links_raw   = spark.read.schema(links_schema).option("header", True).csv(links_path)
df_tags_raw    = spark.read.schema(tags_schema).option("header", True).csv(tags_path)
df_ratings_raw = spark.read.schema(ratings_schema).option("header", True).csv(ratings_path)

print("raw counts:",
      df_movies_raw.count(),
      df_links_raw.count(),
      df_tags_raw.count(),
      df_ratings_raw.count())

df_movies_raw.show(5, truncate=False)
df_links_raw.show(5, truncate=False)
df_tags_raw.show(5, truncate=False)
df_ratings_raw.show(5, truncate=False)

## Clean movies (genres array + year + title_clean)

In [0]:
from pyspark.sql import functions as F

df_movies_silver = (
    df_movies_raw
    .dropDuplicates(["movieId"])
    .withColumn("title", F.trim("title"))
    .withColumn("genres", F.trim("genres"))
    .withColumn("genres_array", F.split(F.coalesce("genres", F.lit("")), "\\|"))
    .withColumn("genres_array", F.expr("filter(genres_array, x -> x != '' and x != '(no genres listed)')"))
    .withColumn("movie_year", F.regexp_extract("title", r"\((\d{4})\)\s*$", 1).cast("int"))
    .withColumn("title_clean", F.regexp_replace("title", r"\s*\(\d{4}\)\s*$", ""))
)

df_movies_silver.select("movieId","title","title_clean","movie_year","genres_array").show(5, truncate=False)


## Clean links (dedupe)

In [0]:
df_links_silver = df_links_raw.dropDuplicates(["movieId"])
print("links distinct movieId:", df_links_silver.select("movieId").distinct().count())
df_links_silver.show(5, truncate=False)


## Clean tags (timestamp + orphan check)

In [0]:
df_tags_silver = (
    df_tags_raw
    .dropDuplicates(["userId","movieId","timestamp","tag"])
    .withColumn("tag_ts", F.from_unixtime("timestamp").cast("timestamp"))
)

orphan_tags = df_tags_silver.join(df_movies_silver.select("movieId"), on="movieId", how="left_anti")
print("orphan tags:", orphan_tags.count())
orphan_tags.show(10, truncate=False)


## Clean ratings (timestamp + domain + orphan check)

In [0]:
df_ratings_silver = (
    df_ratings_raw
    .dropDuplicates(["userId","movieId","timestamp"])
    .withColumn("rating_ts", F.from_unixtime("timestamp").cast("timestamp"))
    .filter(F.col("rating").between(0.5, 5.0))
)

orphan_ratings = df_ratings_silver.join(df_movies_silver.select("movieId"), on="movieId", how="left_anti")
print("orphan ratings:", orphan_ratings.count())
orphan_ratings.show(10, truncate=False)

## Build “movie master” (movies + links)

In [0]:
df_movies_master = df_movies_silver.join(df_links_silver, on="movieId", how="left")
print("movie master rows:", df_movies_master.count())
df_movies_master.select("movieId","title_clean","imdbId","tmdbId","genres_array").show(5, truncate=False)


# Gold transformations

## Performance setup (important with 32M ratings)

In [0]:
from pyspark.sql import functions as F

# Tune partitions for local machine (adjust 8/16/32 based on CPU)
spark.conf.set("spark.sql.shuffle.partitions", "16")

# Cache ratings; we’ll reuse it multiple times
df_ratings_silver = df_ratings_silver.select("userId", "movieId", "rating", "rating_ts").cache()
_ = df_ratings_silver.count()   # materialize cache


## Gold: fact_ratings (ALS-ready)

In [0]:
fact_ratings = (
    df_ratings_silver
    .select("userId", "movieId", "rating")
)

print("fact_ratings rows:", fact_ratings.count())
fact_ratings.show(5)


## dev sampling for dim_users

In [0]:
from pyspark.sql import functions as F

# 1% sample for local dev (tune 0.005–0.05 depending on laptop)
ratings_dev = df_ratings_silver.sample(False, 0.01, seed=42).cache()
_ = ratings_dev.count()


## Gold: dim_users (behavior profile)

In [0]:
dim_users_dev = (
    ratings_dev
    .groupBy("userId")
    .agg(
        F.count("*").alias("num_ratings"),
        F.avg("rating").alias("avg_rating"),
        F.min("rating_ts").alias("first_rating_ts"),
        F.max("rating_ts").alias("last_rating_ts"),
        F.countDistinct("movieId").alias("distinct_movies_rated"),
    )
    .withColumn("is_power_user", F.col("num_ratings") >= F.lit(200))
)
dim_users_dev.orderBy(F.desc("num_ratings")).limit(10).show(truncate=False)

## Movie rating stats (use the same ratings_dev sample)

In [0]:
from pyspark.sql import functions as F

movie_rating_stats_dev = (
    ratings_dev
    .groupBy("movieId")
    .agg(
        F.count("*").alias("num_ratings"),
        F.avg("rating").alias("avg_rating")
    )
)

movie_rating_stats_dev.orderBy(F.desc("num_ratings")).show(10, truncate=False)


## Build dim_movies_enriched (movies + links + rating stats)

In [0]:
df_movies_master = df_movies_silver.join(df_links_silver, on="movieId", how="left")

dim_movies_enriched_dev = (
    df_movies_master
    .join(movie_rating_stats_dev, on="movieId", how="left")
    .fillna({"num_ratings": 0})
)

dim_movies_enriched_dev.select(
    "movieId", "title_clean", "movie_year", "genres_array", "imdbId", "tmdbId", "num_ratings", "avg_rating"
).orderBy(F.desc("num_ratings")).show(10, truncate=False)


In [0]:
top_movies = (
    dim_movies_enriched_dev
    .orderBy(F.desc("num_ratings"))
    .select("movieId", "title_clean", "movie_year", "imdbId", "tmdbId", "num_ratings")
    .limit(500)
)

top_movies.show(20, truncate=False)



In [0]:
top500_pairs = [
    (int(r["movieId"]), int(r["imdbId"]))
    for r in top_movies.select("movieId", "imdbId").collect()
    if r["imdbId"] is not None
]

len(top500_pairs), top500_pairs[:5]

## scraping

In [0]:
# If not installed yet (run once):
# %pip install requests beautifulsoup4 lxml tenacity

import json
import re
import time
import requests
from bs4 import BeautifulSoup
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

from pyspark.sql import functions as F


## Select top 500 movies to scrape

In [0]:
top_movies_5 = (
    dim_movies_enriched_dev
    .orderBy(F.desc("num_ratings"))
    .select("movieId", "title_clean", "movie_year", "imdbId", "tmdbId", "num_ratings")
    .limit(500)
    .cache()
)

print("Top 500 preview:")
top_movies_5.show(truncate=False)

top5_pairs = [
    (int(r["movieId"]), int(r["imdbId"]))
    for r in top_movies_5.select("movieId", "imdbId").collect()
    if r["imdbId"] is not None
]

print("Pairs (movieId, imdbId):", top5_pairs)


## IMDb scraper (director/title/poster from JSON-LD + budget)

In [0]:
SESSION = requests.Session()
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

_money_re = re.compile(r"\$[\d,]+")

@retry(
    reraise=True,
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    retry=retry_if_exception_type(requests.RequestException),
)
def fetch_imdb_html(imdb_id: int) -> str:
    imdb7 = str(imdb_id).zfill(7)
    url = f"https://www.imdb.com/title/tt{imdb7}/"
    resp = SESSION.get(url, headers=HEADERS, timeout=25)
    resp.raise_for_status()
    return resp.text

def parse_jsonld(soup: BeautifulSoup) -> dict:
    script = soup.find("script", type="application/ld+json")
    if not script or not script.string:
        return {}
    try:
        data = json.loads(script.string)
        return data if isinstance(data, dict) else {}
    except json.JSONDecodeError:
        return {}

def extract_budget_best_effort(soup: BeautifulSoup):
    """
    Tries to find 'Budget' label and grab a nearby '$x,xxx,xxx' token.
    This is heuristic and may return None frequently.
    """
    node = soup.find(string=re.compile(r"^\s*Budget\s*$", re.IGNORECASE))
    if not node:
        return None

    # Walk up a few ancestors and search for $ amounts
    container = node.parent
    for _ in range(6):
        if container is None:
            break
        text = container.get_text(" ", strip=True)
        m = _money_re.search(text)
        if m:
            return m.group(0)
        container = container.parent

    return None

def scrape_imdb_movie(movie_id: int, imdb_id: int) -> dict:
    out = {
        "movieId": int(movie_id),
        "imdbId": int(imdb_id),
        "title": None,
        "director": None,
        "budget": None,        # string like "$50,000,000" if found
        "poster_url": None,
        "status": "ok",
        "source": "imdb"
    }

    try:
        html = fetch_imdb_html(imdb_id)
        soup = BeautifulSoup(html, "lxml")

        # JSON-LD: title, director, poster
        data = parse_jsonld(soup)

        out["title"] = data.get("name")

        director = data.get("director")
        if isinstance(director, dict):
            out["director"] = director.get("name")
        elif isinstance(director, list) and director:
            out["director"] = director[0].get("name")

        image = data.get("image")
        if isinstance(image, str):
            out["poster_url"] = image

        # Budget: heuristic
        out["budget"] = extract_budget_best_effort(soup)

    except requests.HTTPError as e:
        code = e.response.status_code if e.response is not None else "na"
        out["status"] = f"http:{code}"
    except Exception as e:
        out["status"] = f"err:{type(e).__name__}"

    return out


## Scrape top 5

In [0]:
results = []
sleep_sec = 1.0  # be polite; reduce risk of blocks

for i, (movie_id, imdb_id) in enumerate(top5_pairs, start=1):
    rec = scrape_imdb_movie(movie_id, imdb_id)
    results.append(rec)
    print(f"{i}/500 -> {rec['movieId']} status={rec['status']} director={rec['director']} budget={rec['budget']}")
    time.sleep(sleep_sec)

results


## Save to JSONL

In [0]:
import html

out_path = "scraped_metadata.jsonl"  # rename from top5 if this is full 500

with open(out_path, "w", encoding="utf-8") as f:
    for r in results:
        # Clean HTML entities
        if r.get("title"):
            r["title"] = html.unescape(r["title"])
        if r.get("director"):
            r["director"] = html.unescape(r["director"])

        f.write(json.dumps(r) + "\n")

print("✅ saved:", out_path)


## Load the scraped JSONL back

In [0]:
scraped_top5_df = spark.read.json("scraped_metadata_top500.jsonl")
scraped_top5_df.show(truncate=False)


## Recreate dim_movies_enriched_dev

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, LongType, DoubleType

# schemas (ensure these match yours)
movies_schema = StructType([
    StructField("movieId", IntegerType(), False),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True),
])
links_schema = StructType([
    StructField("movieId", IntegerType(), False),
    StructField("imdbId", IntegerType(), True),
    StructField("tmdbId", IntegerType(), True),
])
ratings_schema = StructType([
    StructField("userId", IntegerType(), False),
    StructField("movieId", IntegerType(), False),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", LongType(), True),
])

movies_path   = r"..\data\raw\movies.csv"
links_path    = r"..\data\raw\links.csv"
ratings_path  = r"..\data\raw\ratings.csv"

df_movies_raw  = spark.read.schema(movies_schema).option("header", True).csv(movies_path)
df_links_raw   = spark.read.schema(links_schema).option("header", True).csv(links_path)
df_ratings_raw = spark.read.schema(ratings_schema).option("header", True).csv(ratings_path)

df_movies_silver = (
    df_movies_raw.dropDuplicates(["movieId"])
    .withColumn("title", F.trim("title"))
    .withColumn("genres", F.trim("genres"))
    .withColumn("genres_array", F.split(F.coalesce("genres", F.lit("")), "\\|"))
    .withColumn("genres_array", F.expr("filter(genres_array, x -> x != '' and x != '(no genres listed)')"))
    .withColumn("movie_year", F.regexp_extract("title", r"\((\d{4})\)\s*$", 1).cast("int"))
    .withColumn("title_clean", F.regexp_replace("title", r"\s*\(\d{4}\)\s*$", ""))
)

df_links_silver = df_links_raw.dropDuplicates(["movieId"])
df_movies_master = df_movies_silver.join(df_links_silver, on="movieId", how="left")

ratings_dev = (
    df_ratings_raw
    .select("movieId", "rating")
    .sample(False, 0.01, seed=42)
    .cache()
)
_ = ratings_dev.count()

movie_stats_dev = (
    ratings_dev.groupBy("movieId")
    .agg(
        F.count("*").alias("num_ratings"),
        F.avg("rating").alias("avg_rating")
    )
)

dim_movies_enriched_dev = (
    df_movies_master
    .join(movie_stats_dev, on="movieId", how="left")
    .fillna({"num_ratings": 0})
)


## fixed join

In [0]:
from pyspark.sql import functions as F

scraped_top5_df2 = scraped_top5_df.select(
    "movieId",
    F.col("title").alias("scraped_title"),
    "director",
    "budget",
    "poster_url",
    "status"
)

movies_enriched_top5 = dim_movies_enriched_dev.join(scraped_top5_df2, on="movieId", how="left")

movies_enriched_top5.select(
    "movieId", "title_clean", "scraped_title", "director", "budget", "poster_url", "status"
).orderBy(F.desc("num_ratings")).show(20, truncate=False)


## Clean budget into numeric format

In [0]:
from pyspark.sql import functions as F

scraped_df_clean = (
    scraped_top5_df
    .withColumn("budget_clean",
        F.regexp_replace("budget", "[$,]", "").cast("double")
    )
)


## Build Final Gold Dimension

In [0]:
scraped_selected = scraped_df_clean.select(
    "movieId",
    F.col("title").alias("scraped_title"),
    "director",
    "budget_clean",
    "poster_url"
)

dim_movies_enriched_final = (
    dim_movies_enriched_dev
    .join(scraped_selected, on="movieId", how="left")
)


In [0]:
dim_movies_enriched_final.select(
    "movieId",
    "title_clean",
    "director",
    "budget_clean",
    "num_ratings",
    "avg_rating"
).orderBy(F.desc("num_ratings")).show(10, truncate=False)


## Highest Rated Director (Showcase Query)

In [0]:
director_ratings = (
    dim_movies_enriched_final
    .filter(F.col("director").isNotNull())
    .filter(F.col("num_ratings") >= 50)  # threshold to avoid noise
    .groupBy("director")
    .agg(
        F.avg("avg_rating").alias("director_avg_rating"),
        F.count("*").alias("num_movies"),
        F.sum("num_ratings").alias("total_ratings")
    )
    .orderBy(F.desc("director_avg_rating"))
)

director_ratings.show(20, truncate=False)


## Build the training dataset (ALS input)

In [0]:
from pyspark.sql import functions as F

fact_ratings = (
    df_ratings_silver
    .select("userId", "movieId", "rating")
)

fact_ratings.printSchema()
fact_ratings.show(5)


In [0]:
train_df = (
    fact_ratings
    .sample(False, 0.02, seed=42)   # 2% sample
    .cache()
)

print("train rows:", train_df.count())


## Train/Test split

In [0]:
train, test = train_df.randomSplit([0.8, 0.2], seed=42)

print("train:", train.count())
print("test :", test.count())


## Train ALS model

In [0]:
from pyspark.ml.recommendation import ALS

als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    implicitPrefs=False,
    coldStartStrategy="drop",   # critical for sparsity
    nonnegative=True,
    rank=20,
    regParam=0.1,
    maxIter=10
)

als_model = als.fit(train)

print("✅ ALS training completed")


## Evaluate model (RMSE)

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

predictions = als_model.transform(test)

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(predictions)
print("RMSE:", rmse)


## Generate Top-10 Recommendations

In [0]:
user_recs = als_model.recommendForAllUsers(10)

user_recs.show(5, truncate=False)


## Make recommendations human-readable

In [0]:
recs_flat = (
    user_recs
    .select("userId", F.explode("recommendations").alias("rec"))
    .select(
        "userId",
        F.col("rec.movieId").alias("movieId"),
        F.col("rec.rating").alias("pred_score")
    )
)

recs_with_titles = (
    recs_flat
    .join(
        dim_movies_enriched_final.select(
            "movieId", "title_clean", "director", "poster_url"
        ),
        on="movieId",
        how="left"
    )
)

recs_with_titles.show(20, truncate=False)
